# 🔵 Clustering — Agrupando Barrios por Perfil Socioeconómico
### DiploDatos 2026 — FAMAF / Universidad Nacional de Córdoba

En este notebook vamos a aplicar **K-Means** para agrupar los barrios de Córdoba según sus características socioeconómicas y de acceso a servicios.

**Objetivo:** Identificar perfiles de barrio (p.ej: «barrio vulnerable sin servicios», «barrio de clase media con buena infraestructura», etc.)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 5)
plt.style.use('seaborn-v0_8-whitegrid')

df = pd.read_csv('../data/processed/dataset_final_v6.csv')
print(f'Dataset: {len(df)} barrios, {df.shape[1]} columnas')
df.head(3)

## 1. Selección de variables y preprocesamiento

In [ ]:
# Variables para el clustering
FEATURES = [
    'pct_nbi',           # nivel de pobreza estructural
    'escuelas_total',    # acceso a educación
    'escuelas_estatales',
    'escuelas_privadas',
    'centros_salud',     # acceso a salud
    'paradas_colectivo', # acceso a transporte
    'luminarias_reportes',
    'comisarias',
]
FEATURES = [f for f in FEATURES if f in df.columns]

# Trabajar solo con barrios que tienen datos de NBI (variable clave)
df_cluster = df.dropna(subset=['pct_nbi'])[['barrio'] + FEATURES].copy()
df_cluster[FEATURES] = df_cluster[FEATURES].fillna(0)
print(f'Barrios usados para clustering: {len(df_cluster)}')

# Normalizar (KMeans es sensible a la escala)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_cluster[FEATURES])
print(f'Variables: {FEATURES}')

## 2. Método del Codo — ¿Cuántos clusters usar?

In [ ]:
inercias = []
silhouettes = []
K_range = range(2, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inercias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(K_range, inercias, 'bo-', linewidth=2, markersize=7)
axes[0].set_xlabel('Número de clusters (K)')
axes[0].set_ylabel('Inercia (suma de distancias cuadradas)')
axes[0].set_title('Método del Codo', fontsize=12, fontweight='bold')
axes[0].set_xticks(K_range)

axes[1].plot(K_range, silhouettes, 'rs-', linewidth=2, markersize=7)
axes[1].set_xlabel('Número de clusters (K)')
axes[1].set_ylabel('Silhouette Score (más alto = mejor)')
axes[1].set_title('Silhouette Score', fontsize=12, fontweight='bold')
axes[1].set_xticks(K_range)

plt.tight_layout()
plt.savefig('../tmp/clustering_codo.png', dpi=130, bbox_inches='tight')
plt.show()

k_optimo = K_range[silhouettes.index(max(silhouettes))]
print(f'K con mejor Silhouette: {k_optimo}')

## 3. K-Means con K=5 (buen equilibrio entre interpretabilidad y calidad)

In [ ]:
K = 5
km_final = KMeans(n_clusters=K, random_state=42, n_init=10)
df_cluster['cluster'] = km_final.fit_predict(X_scaled)

# Perfil de cada cluster
perfil = df_cluster.groupby('cluster')[FEATURES].mean().round(2)
perfil['n_barrios'] = df_cluster.groupby('cluster').size()
print(f'Silhouette Score con K={K}: {silhouette_score(X_scaled, df_cluster["cluster"]):.3f}')
print()
print('Perfil de cada cluster (valores medios):')
print(perfil.to_string())

## 4. Visualización con PCA (reducción a 2 dimensiones)

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

COLORES = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6']
NOMBRES_CLUSTER = [
    'Cluster 0', 'Cluster 1', 'Cluster 2', 'Cluster 3', 'Cluster 4'
]

fig, ax = plt.subplots(figsize=(11, 7))
for c in range(K):
    mask = df_cluster['cluster'] == c
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
               c=COLORES[c], s=60, alpha=0.7, label=f'Cluster {c} (n={mask.sum()})')

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% varianza)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% varianza)')
ax.set_title('Clustering de Barrios de Córdoba (K-Means con PCA)', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('../tmp/clustering_pca.png', dpi=130, bbox_inches='tight')
plt.show()

var_explicada = pca.explained_variance_ratio_.sum() * 100
print(f'Varianza explicada por PC1+PC2: {var_explicada:.1f}%')

## 5. Perfil detallado de cada cluster

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6))

n_features = len(FEATURES)
x = np.arange(n_features)
width = 0.15

for i, c in enumerate(range(K)):
    valores = perfil.loc[c, FEATURES]
    valores_norm = (valores - valores.min()) / (valores.max() - valores.min() + 1e-9)
    ax.bar(x + i * width, valores_norm, width, label=f'Cluster {c}', color=COLORES[i], alpha=0.85)

labels_ejes = [f.replace('_', '\n') for f in FEATURES]
ax.set_xticks(x + width * (K - 1) / 2)
ax.set_xticklabels(labels_ejes, fontsize=8)
ax.set_ylabel('Valor normalizado (0-1)')
ax.set_title('Perfil de cada Cluster (valores normalizados)', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('../tmp/clustering_perfil.png', dpi=130, bbox_inches='tight')
plt.show()

## 6. Barrios típicos de cada cluster

In [ ]:
# Exportar resultado del clustering al dataset
df_resultado = df.merge(df_cluster[['barrio', 'cluster']], on='barrio', how='left')
df_resultado.to_csv('../data/processed/dataset_final_v6_clusters.csv', index=False)

# Mostrar barrios representativos (los más cercanos al centroide)
from scipy.spatial.distance import cdist
centroides_pca = np.array([X_pca[df_cluster['cluster'] == c].mean(axis=0) for c in range(K)])
distancias = cdist(X_pca, centroides_pca)

print('Barrios más representativos de cada cluster:')
for c in range(K):
    mask = df_cluster['cluster'] == c
    idx_cluster = df_cluster[mask].index
    dist_cluster = distancias[idx_cluster, c]
    top5 = df_cluster.loc[idx_cluster[np.argsort(dist_cluster)[:5]], 'barrio'].tolist()
    nbi_medio = perfil.loc[c, 'pct_nbi']
    n = perfil.loc[c, 'n_barrios']
    print(f'  Cluster {c} [{n} barrios, NBI medio={nbi_medio:.1f}%]: {top5}')

## 📝 Conclusiones del Clustering

1. **K-Means con K=5** permite identificar perfiles diferenciados de barrios.
2. Los clusters capturan diferencias en NBI, escuelas y transporte.
3. El resultado se guardó en `data/processed/dataset_final_v6_clusters.csv`.
4. **Para el siguiente notebook:** construimos un modelo predictivo para estimar el NBI de un barrio.